# Acquisizione su griglia di punti X-Y da file (laser + oscilloscopio + motori PI)

Per ogni punto di una lista X-Y letta da `punti.txt`: si muovono i due motori PI C-863, poi si fa **una scansione completa** del laser sincronizzata con lo scope (stessa sequenza di `acquisizione_laser_scope.ipynb`, quella che funziona), e si salva un CSV per punto.

- Motore **X** = `ASRL4` (COM4), motore **Y** = `ASRL5` (COM5). Limiti 0-150 mm.
- **Chiudi PI MicroMove** prima di partire: la porta seriale può essere usata da un solo programma.
- I punti stanno in `punti.txt` (nella cartella del notebook, nome modificabile in cella 1): una riga per punto, due colonne `x y` in mm (posizioni assolute) separate da spazi; le righe che iniziano con `#` sono ignorate. I punti non devono essere equidistanti e vengono scansionati **nell'ordine del file**. Finché il file non ha punti, nessun motore si muove.
- Per una matrice a meandro intorno alla membrana usa `acquisizione_matrice_xy.ipynb`.

Ordine: celle 1-4 una volta; la cella 5 legge il file e mostra l'anteprima dei punti (**non muove niente**); la cella 6 li scansiona tutti e chiede conferma prima di muovere i motori.

Attenzione allo spazio su disco: ogni CSV di un'acquisizione (5 milioni di campioni) pesa circa 240 MB.

In [ ]:
# Cella 1 - configurazione e funzioni (scope, laser, griglia)
import os
import sys
import time
import datetime

import numpy as np
import pyvisa

OSC_VISA = "USB::0x0699::0x0408::C028813::INSTR"
OSC_TIMEOUT_MS = 30000

CHANNELS = ["CH1", "CH2"]

LASER_WAVELENGTH_START = 1520.0  # nm, come trigger_scan.m
LASER_WAVELENGTH_STOP = 1570.0   # nm, come trigger_scan.m

POLL_INTERVAL_S = 0.1
CHANNEL_READ_PAUSE_S = 0.1

# ---------------- punti X-Y (in mm, limiti motori 0-150) ----------------
FILE_PUNTI = "punti.txt"   # colonne: x y (mm); righe con # ignorate

ASSESTAMENTO_S = 1.0           # pausa dopo il movimento, prima di acquisire
ATTESA_RITORNO_LASER_S = 15.0  # tra due scan, perché il laser torni a 1520 nm (CH1 basso)
# --------------------------------------------------------------------------


def crea_cartella_run():
    oggi = datetime.date.today().strftime("%Y-%m-%d")
    base_folder = os.path.join(os.getcwd(), oggi)
    os.makedirs(base_folder, exist_ok=True)
    run_esistenti = [d for d in os.listdir(base_folder) if d.startswith("run_")]
    run_folder = os.path.join(base_folder, f"run_{len(run_esistenti) + 1:03d}")
    os.makedirs(run_folder)
    return run_folder


def configura_scope(osc):
    osc.write("HORizontal:MODE MANUAL")
    rec_len = int(osc.query("HORizontal:RECORDLENGTH?"))
    osc.write("DATA:START 1")
    osc.write(f"DATA:STOP {rec_len}")
    osc.write("DATA:WIDTH 1")
    osc.write("DATA:ENCdg ASCii")


def arm_acquisition(osc):
    osc.write("ACQ:STOPAFTER SEQUENCE")
    osc.write("ACQ:STATE RUN")
    # Lo scan parte solo quando lo scope e' READY (finito il pre-trigger).
    print("Attendo che lo scope finisca il pre-trigger (-> READY)...")
    while osc.query("TRIGger:STATE?").strip().upper() != "READY":
        time.sleep(POLL_INTERVAL_S)
    print("Scope pronto ad accettare il trigger.")


def wait_for_acquisition(osc):
    print("Attendo che l'oscilloscopio completi l'acquisizione triggerata...")
    while True:
        if int(osc.query("ACQ:STATE?")) == 0:
            print("Acquisizione completata!")
            break
        time.sleep(POLL_INTERVAL_S)


def acquire_channel(osc, source="CH1"):
    osc.write(f"DATA:SOURCE {source}")
    ymult = float(osc.query("WFMPRE:YMULT?"))
    yoff = float(osc.query("WFMPRE:YOFF?"))
    yzero = float(osc.query("WFMPRE:YZERO?"))
    print(f"{source}: YMULT={ymult}, YOFF={yoff}, YZERO={yzero}")
    raw = osc.query("CURVE?")
    ydata = np.array(raw.split(","), dtype=float)
    return (ydata - yoff) * ymult + yzero

In [ ]:
# Cella 2 - connessione all'oscilloscopio (una volta per sessione)
rm = pyvisa.ResourceManager()
osc = rm.open_resource(OSC_VISA)
osc.timeout = OSC_TIMEOUT_MS
configura_scope(osc)
print("Oscilloscopio connesso e configurato.")

In [ ]:
# Cella 3 - laser: carica la DLL e apre il dispositivo (una volta per sessione)
USBADDR = 1
DEVICE_ID = 0x100A  # hex2dec('100A')

dll_path = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "scripts", "laser", "UsbDllWrap.dll"))
if not os.path.isfile(dll_path):
    raise RuntimeError(f"UsbDllWrap.dll non trovata in {dll_path} (il notebook va aperto da Chiara_Bru/runs_oscilloscope)")
print(f"DLL trovata: {dll_path}")
sys.path.append(os.path.dirname(dll_path))

import clr
clr.AddReference("UsbDllWrap")
import Newport
from System.Text import StringBuilder

NP_USB = Newport.USBComm.USB()

esito = NP_USB.OpenDevices(DEVICE_ID)
print(f"OpenDevices({hex(DEVICE_ID)}) -> {esito}")
querydata = StringBuilder(64)


def laser_query(comando):
    esito = NP_USB.Query(USBADDR, comando, querydata)
    risposta = querydata.ToString()
    print(f"Query {comando!r} -> esito={esito}, risposta={risposta!r}")
    return risposta


def laser_write(comando):
    esito = NP_USB.Write(USBADDR, comando)
    print(f"Write {comando!r} -> esito={esito}")


laser_query("*IDN?")
laser_query("OUTP:STAT?")

In [ ]:
# Cella 4 - motori PI C-863: X = ASRL4, Y = ASRL5 (una volta per sessione)
MOTORE_X = "ASRL4::INSTR"
MOTORE_Y = "ASRL5::INSTR"
PI_BAUD = 19200
PI_AXIS = "1"
PI_TIMEOUT_MS = 2000
PI_TIMEOUT_MOVIMENTO_S = 60
PI_TOLLERANZA_MM = 0.01  # valore usato nella prova sul laboratorio (i motori arrivano entro circa 10 um)


def pi_apri(porta):
    inst = rm.open_resource(porta)
    inst.baud_rate = PI_BAUD
    inst.timeout = PI_TIMEOUT_MS
    inst.read_termination = "\n"
    inst.write_termination = "\n"
    return inst


def pi_query(inst, comando):
    return inst.query(comando).strip()


def pi_valore(inst, comando):
    """Estrae il numero da risposte tipo '1=+0012.3456'."""
    return float(pi_query(inst, comando).split("=")[-1])


def pi_limiti(inst):
    return pi_valore(inst, f"TMN? {PI_AXIS}"), pi_valore(inst, f"TMX? {PI_AXIS}")


def pi_muovi(inst, nome, target):
    """Movimento assoluto in mm, con controllo dei limiti prima di muovere."""
    tmin, tmax = pi_limiti(inst)
    if not (tmin <= target <= tmax):
        raise ValueError(f"Motore {nome}: target {target} mm fuori dai limiti {tmin}..{tmax} mm")
    inst.write(f"MOV {PI_AXIS} {target:.4f}")
    time.sleep(0.1)
    t0 = time.time()
    while True:
        in_target = pi_query(inst, f"ONT? {PI_AXIS}").endswith("1")
        pos = pi_valore(inst, f"POS? {PI_AXIS}")
        if in_target and abs(pos - target) <= PI_TOLLERANZA_MM:
            break
        if time.time() - t0 > PI_TIMEOUT_MOVIMENTO_S:
            raise TimeoutError(f"Motore {nome}: non arrivato a {target} mm (posizione {pos} mm)")
        time.sleep(0.1)
    errore = int(pi_query(inst, "ERR?"))
    if errore != 0:
        raise RuntimeError(f"Motore {nome}: ERR? = {errore} dopo il movimento a {target} mm")
    print(f"Motore {nome}: a {pos:.4f} mm")


motore_x = pi_apri(MOTORE_X)
motore_y = pi_apri(MOTORE_Y)

for nome, inst in (("X", motore_x), ("Y", motore_y)):
    errore_residuo = pi_query(inst, "ERR?")  # legge e azzera un eventuale errore vecchio
    tmin, tmax = pi_limiti(inst)
    print(f"Motore {nome}: {pi_query(inst, '*IDN?')}")
    print(f"   posizione {pi_valore(inst, f'POS? {PI_AXIS}')} mm | limiti {tmin}..{tmax} mm | "
          f"servo {pi_query(inst, f'SVO? {PI_AXIS}')} | referenziato {pi_query(inst, f'FRF? {PI_AXIS}')} | "
          f"errore residuo letto: {errore_residuo}")

In [ ]:
# Cella 5 - legge i punti dal file e mostra l'ANTEPRIMA (NON muove niente)
percorso_punti = os.path.join(os.getcwd(), FILE_PUNTI)
if not os.path.isfile(percorso_punti):
    raise FileNotFoundError(f"File dei punti non trovato: {percorso_punti}")

punti_file = np.loadtxt(percorso_punti, comments="#", ndmin=2)
if punti_file.size == 0:
    raise RuntimeError(f"{FILE_PUNTI} non contiene punti: aggiungi una riga 'x y' (mm) per punto")
if punti_file.shape[1] != 2:
    raise ValueError(f"{FILE_PUNTI}: servono esattamente 2 colonne (x y), trovate {punti_file.shape[1]}")

PUNTI = [(float(x), float(y)) for x, y in punti_file]

for nome, inst, colonna in (("X", motore_x, punti_file[:, 0]), ("Y", motore_y, punti_file[:, 1])):
    tmin, tmax = pi_limiti(inst)
    if colonna.min() < tmin or colonna.max() > tmax:
        raise ValueError(f"Punti {nome} fuori dai limiti {tmin}..{tmax} mm")

print(f"{len(PUNTI)} punti letti da {FILE_PUNTI}, limiti rispettati.")
for i, (x, y) in enumerate(PUNTI):
    print(f"  {i + 1:3d}: X={x:.4f}  Y={y:.4f}")
print(f"Spazio su disco stimato: circa {len(PUNTI) * 0.24:.1f} GB")

In [ ]:
# Cella 6 - ESEGUE la griglia: per ogni punto muove X e Y, poi una scansione completa
if input(f"Muovere i motori ed eseguire {len(PUNTI)} scansioni? scrivi 'si' per confermare: ").strip().lower() != "si":
    raise RuntimeError("Annullato: nessun movimento eseguito.")

run_folder = crea_cartella_run()
print(f"I dati saranno salvati in: {run_folder}")
registro_path = os.path.join(run_folder, "griglia.csv")
with open(registro_path, "w") as f:
    f.write("indice,x_mm,y_mm,file\n")

try:
    for idx, (x, y) in enumerate(PUNTI):
        print(f"\n=== Punto {idx + 1}/{len(PUNTI)}: X={x:.3f} mm, Y={y:.3f} mm ===")
        pi_muovi(motore_x, "X", x)
        pi_muovi(motore_y, "Y", y)
        time.sleep(ASSESTAMENTO_S)

        # stessa sequenza dell'altro notebook (acquisizione_laser_scope.ipynb)
        laser_write(f"SOUR:WAVE:START {LASER_WAVELENGTH_START}")
        laser_write(f"SOUR:WAVE:STOP {LASER_WAVELENGTH_STOP}")

        arm_acquisition(osc)
        laser_write("OUTP:SCAN:START")

        wait_for_acquisition(osc)
        time.sleep(1.0)

        data = {}
        for ch in CHANNELS:
            print(f"Leggo {ch}...")
            data[ch] = acquire_channel(osc, ch)
            time.sleep(CHANNEL_READ_PAUSE_S)

        nome_file = f"punto_{idx:03d}_x{x:.3f}_y{y:.3f}.csv"
        min_len = min(len(v) for v in data.values())
        all_cols = np.column_stack([data[ch][:min_len] for ch in CHANNELS])
        np.savetxt(os.path.join(run_folder, nome_file), all_cols, delimiter=",", header=",".join(CHANNELS), comments="")
        with open(registro_path, "a") as f:
            f.write(f"{idx},{x:.4f},{y:.4f},{nome_file}\n")
        print(f"Salvato: {nome_file}")

        if idx < len(PUNTI) - 1:
            print(f"Attendo {ATTESA_RITORNO_LASER_S:.0f} s che il laser torni a {LASER_WAVELENGTH_START} nm...")
            time.sleep(ATTESA_RITORNO_LASER_S)
except BaseException:
    print("\nInterrotto: provo a fermare lo scan del laser.")
    try:
        laser_write("OUTP:SCAN:STOP")
    except Exception as e:
        print(f"Impossibile fermare lo scan: {e}")
    raise

print(f"\nGriglia completata: {len(PUNTI)} punti in {run_folder}")